# 06 · Aplicación financiera con Streamlit desde Google Colab

**Financial Analytics Lab · Python · Streamlit · yfinance**

**Pregunta:** ¿cómo construir y probar una aplicación interactiva sin instalar programas en el computador?

Este módulo se desarrolla íntegramente en **Google Colab**. Allí instalaremos temporalmente las bibliotecas, recuperaremos el código desde GitHub y ejecutaremos Streamlit. El proyecto tiene dos versiones de una misma aplicación: inicial (S27) y ampliada (S33); su defensa individual (S34) se acordará conforme a las condiciones académicas autorizadas.

**Importante:** Colab no es alojamiento permanente. El acceso a la aplicación funciona únicamente mientras la sesión está activa y depende de las funciones de proxy disponibles. Conserva el código y las evidencias en GitHub.

## 1 · Preparar Colab

Ejecuta esta celda para instalar las bibliotecas **en la sesión de Colab**, no en tu computador. Si se reinicia el entorno, tendrás que ejecutarla otra vez.

In [ ]:
%pip -q install streamlit yfinance pandas numpy matplotlib requests

## 2 · Recuperar el código desde GitHub

El código publicado en GitHub se descargará a la carpeta temporal `/content/financial_analytics_lab`. La descarga no modifica el repositorio; para guardar cambios deberás subir los archivos actualizados a GitHub.

In [ ]:
from pathlib import Path
import requests
CARPETA = Path("/content/financial_analytics_lab")
CARPETA.mkdir(parents=True, exist_ok=True)
BASE = "https://raw.githubusercontent.com/vyolete/proyecto_analitica_financiera/main/06_financial_app"
for nombre in ("app.py", "app_final.py"):
    respuesta = requests.get(f"{BASE}/{nombre}", timeout=30)
    respuesta.raise_for_status()
    (CARPETA / nombre).write_text(respuesta.text, encoding="utf-8")
print("Archivos recuperados:", [x.name for x in CARPETA.glob("*.py")])

## 3 · Verificar los indicadores antes de abrir la aplicación

Un cálculo puede probarse con datos hipotéticos, incluso cuando no está disponible Yahoo Finance. Comprueba rendimiento acumulado y drawdown; ambos aparecerán después en la interfaz.

In [ ]:
import pandas as pd
import numpy as np
ejemplo = pd.Series([100.0, 105.0, 102.0, 110.0])
rendimiento = ejemplo.iloc[-1] / ejemplo.iloc[0] - 1
drawdown = (ejemplo / ejemplo.cummax() - 1).min()
assert np.isclose(rendimiento, 0.10)
print(f"Rendimiento acumulado: {rendimiento:.2%}")
print(f"Máximo drawdown: {drawdown:.2%}")

**Tu práctica 1:** modifica el último precio a 95. Interpreta el cambio en el rendimiento y en el drawdown.

## 4 · Iniciar la primera versión (S27)

Streamlit se ejecutará como un proceso del entorno Colab. No necesitas instalar Python ni abrir una terminal local. Si vuelves a ejecutar esta celda, se detendrá el servidor iniciado previamente por el notebook.

In [ ]:
import subprocess
import sys
import time

def iniciar_app(nombre_archivo):
    global servidor_streamlit
    if "servidor_streamlit" in globals() and servidor_streamlit.poll() is None:
        servidor_streamlit.terminate()
        try:
            servidor_streamlit.wait(timeout=5)
        except subprocess.TimeoutExpired:
            servidor_streamlit.kill()
    registro = open(CARPETA / "streamlit.log", "w", encoding="utf-8")
    servidor_streamlit = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", str(CARPETA / nombre_archivo),
         "--server.port", "8501", "--server.address", "0.0.0.0",
         "--server.headless", "true"],
        stdout=registro, stderr=subprocess.STDOUT
    )
    time.sleep(5)
    if servidor_streamlit.poll() is not None:
        print((CARPETA / "streamlit.log").read_text(encoding="utf-8")[-2500:])
        raise RuntimeError("No se pudo iniciar Streamlit.")
    print("Servidor activo:", nombre_archivo)

iniciar_app("app.py")

## 5 · Abrir la aplicación desde Colab

La siguiente celda utiliza la función de acceso a puertos de Google Colab. Puede abrir una ventana nueva. **Esta función puede variar o no estar disponible en determinadas sesiones**; no garantiza una dirección pública permanente. Si falla, consulta el registro del servidor y conserva el código como evidencia.

In [ ]:
from google.colab import output
try:
    output.serve_kernel_port_as_window(8501)
except Exception as error:
    print("No se pudo abrir la ventana:", type(error).__name__)
    print("Revisa el registro y las funciones disponibles en tu sesión de Colab.")

In [ ]:
print("Servidor:", "activo" if servidor_streamlit.poll() is None else "detenido")
print((CARPETA / "streamlit.log").read_text(encoding="utf-8")[-1500:])

## 6 · Analizar la primera versión

**S27:** selecciona un activo y fechas, consulta precios, observa el rendimiento acumulado, la volatilidad histórica por observación y el gráfico de cierres. Identifica los mensajes de validación y explica sus unidades.

**Tu práctica 2:** compara dos activos en momentos distintos y documenta las diferencias.

**Tu práctica 3:** introduce un intervalo de fechas inválido y explica cómo lo gestiona la aplicación.

**Evidencia:** código `app.py`, captura o grabación breve, comprobación manual de un indicador e interpretación financiera. Las capturas no sustituyen el código.

## 7 · Probar la versión ampliada (S33)

La segunda versión del **mismo proyecto** permite comparar varios activos, establecer ponderaciones y estudiar correlaciones, volatilidad y máximo drawdown. Los pesos se interpretan bajo el supuesto de reequilibrio en cada observación.

Ejecuta la siguiente celda y después vuelve a ejecutar la **celda de apertura** de la sección 5.

In [ ]:
iniciar_app("app_final.py")
print("Ahora vuelve a ejecutar la celda de apertura de la sección 5.")

**Tu práctica 4:** selecciona tres activos y asigna ponderaciones que sumen 100 %. Explica cómo cambia el portafolio al modificar los pesos.

**Tu práctica 5:** interpreta la matriz de correlaciones y un episodio de drawdown sin atribuir causalidad ni hacer predicciones.

**Evidencia S33:** código `app_final.py`, comprobación de indicadores, interpretación y registro de mejoras respecto de S27.

## 8 · Guardar las evidencias en GitHub

Los archivos de `/content` desaparecen cuando se pierde la sesión de Colab. Para conservar modificaciones, descarga los archivos y súbelos a tu repositorio mediante **Add file → Upload files**. No necesitas instalar Git ni utilizar comandos en tu computador.

El enlace temporal de Streamlit no constituye una evidencia permanente. Conserva `app.py`, `app_final.py`, `requirements.txt`, un README y una captura o grabación del funcionamiento.

In [ ]:
# OPCIONAL: descargar código modificado desde Colab
# from google.colab import files
# files.download(str(CARPETA / "app.py"))

## 9 · Defensa individual (S34)

La defensa debe permitir explicar el problema financiero, la fuente de datos, las validaciones, las fórmulas, las decisiones de interfaz, las limitaciones y una modificación sencilla del código. La modalidad remota estará sujeta a la autorización académica correspondiente.

**Reto integrador:** documenta la evolución S27 → S33, comprueba al menos dos indicadores, registra un caso de error y presenta una interpretación financiera de 250 a 350 palabras con supuestos y limitaciones.

**Autoevaluación:** ¿puedo ejecutar la aplicación desde Colab?, ¿comprendo la diferencia entre una copia temporal y GitHub?, ¿sé explicar los indicadores?, ¿conservé mis evidencias?

**Fuentes:** [Streamlit](https://docs.streamlit.io/) · [Google Colab](https://colab.research.google.com/) · [pandas](https://pandas.pydata.org/docs/) · [yfinance](https://ranaroussi.github.io/yfinance/).

Los resultados históricos no son recomendaciones de inversión.

## Quiz de cierre · Aplicación financiera Streamlit

**Autoevaluación de preparación · no genera una nota adicional**

Responde individualmente las cinco preguntas. Las respuestas deben ser originales, razonadas y coherentes con el código que ejecutes. No basta con escribir una definición: explica el significado de los resultados cuando se solicite.

**Criterio sugerido:** cada pregunta vale 1 punto (5 puntos en total). Las preguntas 1–2 evalúan comprensión, 3–4 interpretación y 5 aplicación. La docente verificará las respuestas y registrará la nota conforme a la ponderación institucional de cada quiz.

### Pregunta 1 · Comprensión

¿Qué problema resuelve una aplicación financiera interactiva frente a un cuaderno de análisis?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 2 · Comprensión

¿Por qué una aplicación debe validar el rango de fechas y la selección de activos?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 3 · Interpretación

Explica qué función cumple un control `st.selectbox` en Streamlit.

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 4 · Interpretación

Si una descarga de precios devuelve un DataFrame vacío, ¿qué debería comunicar la aplicación al usuario?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 5 · Aplicación

Escribe un fragmento de código Streamlit que muestre un título y permita seleccionar entre dos activos; explica cómo usarías esa selección.

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._



In [ ]:
# Pregunta 5 — solución individual
# Escribe y ejecuta aquí tu código.
